# YSSY Weather Data Processing and ML Dataset Generation

This notebook recreates the YSSY data-processing workflow from the standalone Python scripts. It can run the upstream station-file preparation stages, generate diagnostics, build the aligned weather-station Parquet file, create machine-learning-ready train, validation, and test datasets, and train/evaluate LightGBM forecast models.

The notebook is designed for manual cell-by-cell execution. Upstream rebuilds, diagnostics, training, and plotting are triggered by running the relevant cells or function calls, rather than by global run switches.


## Workflow

1. **Configuration** - Set folders, stations, weather elements, date ranges, and split proportions.
2. **Raw and Intermediate Data Processing** - Define functions matching the standalone scripts: fixed-width raw parsing, station merging, interpolation, wind-component conversion, cropping, and column dropping.
3. **Diagnostics** - Define missing-timestamp checks, availability summaries, and outage-length summaries.
4. **FinalData to Parquet** - Align all weather stations to a regular 30-minute grid and save `aligned_weather_data.parquet`.
5. **ML Dataset Generation** - Build semantic LightGBM-style features and YSSY forecast targets, split chronologically, and save Parquet datasets.
6. **LightGBM Training** - Load prepared Parquet datasets, train a quick multi-output LightGBM baseline, evaluate validation/test performance, and visualise forecast lead-time error.


In [ ]:
# ============================================================================
# [PART 1] Configuration
# ============================================================================

# Cell notes:
# - Defines all paths, station lists, time windows, split proportions, and dependency checks.
# - No modelling or data transformation happens here; this cell only establishes shared constants.
# - `PYARROW_AVAILABLE` gates Parquet read/write; `SCIPY_AVAILABLE` gates spline interpolation.

from pathlib import Path
from datetime import timedelta
import glob
import os
import random
import time

import matplotlib
matplotlib.use('Agg')
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from scipy.interpolate import UnivariateSpline
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False

try:
    import pyarrow  # Required by pandas read_parquet/to_parquet with engine='pyarrow'
    PYARROW_AVAILABLE = True
except ImportError:
    PYARROW_AVAILABLE = False

# Base folders
BASE_DATA_DIR = Path('/home/timekeeper/Documents/Development/BOM-Team/data/YSSY/Data Processing')
RAW_DATA_DIR = BASE_DATA_DIR / 'RawDataC'
PROCESSED_DATA_DIR = BASE_DATA_DIR / 'ProcessedData'
INTERPOLATED_DATA_DIR = BASE_DATA_DIR / 'InterpolatedData0.5'
UV_COMPONENT_DATA_DIR = BASE_DATA_DIR / 'UVComponentData'
SPLINE_INTERPOLATED_DATA_DIR = BASE_DATA_DIR / 'SplineInterpolatedData'
FINAL_DATA_DIR = BASE_DATA_DIR / 'FinalData'
DIAGNOSTICS_DIR = BASE_DATA_DIR / 'Diagnostics'
OUTPUT_DIR = BASE_DATA_DIR

ALIGNED_DATA_PARQUET = BASE_DATA_DIR / 'aligned_weather_data.parquet'

# Station and feature settings
STATIONS = ['BELL', 'MTB', 'YBTH', 'YCNK', 'YSBK', 'YSCN', 'YSNW', 'YSRI', 'YSSY', 'YSWG']
STATIONS_NO_PRESSURE = ['BELL', 'MTB']
TARGET_STATION = 'YSSY'

ALL_FILE_FEATURES = ['air_temp', 'dew_point', 'msl_pressure', 'u_component', 'v_component']
FEATURES_AFTER_DROP_FOR_NO_PRESSURE_STATIONS = ['air_temp', 'dew_point', 'u_component', 'v_component']

# Date and sampling settings
TIMESTEP_MINUTES = 30
TIMESTEP_RESOLUTION = pd.Timedelta(minutes=TIMESTEP_MINUTES)
AVAILABLE_DATA_START_DATE = pd.Timestamp('2000-01-01 00:00:00')
AVAILABLE_DATA_END_DATE = pd.Timestamp('2024-12-31 23:30:00')
LOOKBACK_WINDOW_HOURS = 24
FORECAST_HORIZON_HOURS = 24
NUM_LOOKBACK_STEPS = int(LOOKBACK_WINDOW_HOURS * 60 / TIMESTEP_MINUTES)
NUM_FORECAST_STEPS = int(FORECAST_HORIZON_HOURS * 60 / TIMESTEP_MINUTES)

TRAIN_SIZE_PROPORTION = 0.08
VALIDATION_SIZE_PROPORTION = 0.01
TEST_SIZE_PROPORTION = 0.01

# Interpolation and diagnostics settings from the standalone scripts
LINEAR_INTERPOLATION_ELEMENTS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'max_gust_speed', 'msl_pressure']
SPLINE_INTERPOLATION_ELEMENTS = ['air_temp', 'dew_point', 'msl_pressure', 'u_component', 'v_component']
MAX_GAP_STEPS_FOR_SPLINE = 5
SPLINE_CONTEXT_POINTS = 6
SPLINE_ORDER = 3
SPLINE_SMOOTHING_FACTOR = 1
COLUMNS_TO_DROP_FROM_FINAL = ['max_gust_speed', 'aws_flag', 'wind_dir_recalc']

for folder in [OUTPUT_DIR, DIAGNOSTICS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Configuration loaded')
print(f'Base data folder: {BASE_DATA_DIR}')
print(f'FinalData folder exists: {FINAL_DATA_DIR.exists()}')
print(f'PyArrow available: {PYARROW_AVAILABLE}')
print(f'SciPy available: {SCIPY_AVAILABLE}')


In [ ]:
# ============================================================================
# [PART 2] Date Boundaries and Shared Helpers
# ============================================================================

# Cell notes:
# - Calculates chronological train/validation/test boundaries from the configured proportions.
# - Provides helper functions used throughout the notebook, including cyclical time encoding.
# - Output variables such as `TRAIN_END_DATE`, `VALIDATION_END_DATE`, and `TEST_END_DATE` are reused later.

def require_pyarrow():
    if not PYARROW_AVAILABLE:
        raise ImportError('Parquet output requires pyarrow. Install it with: pip install pyarrow')


def get_station_features(station_name):
    return FEATURES_AFTER_DROP_FOR_NO_PRESSURE_STATIONS if station_name in STATIONS_NO_PRESSURE else ALL_FILE_FEATURES


def calculate_split_boundaries():
    all_timestamps = pd.date_range(
        start=AVAILABLE_DATA_START_DATE,
        end=AVAILABLE_DATA_END_DATE,
        freq=TIMESTEP_RESOLUTION,
    )
    total_timesteps = len(all_timestamps)

    requested_train = int(np.floor(total_timesteps * TRAIN_SIZE_PROPORTION))
    requested_validation = int(np.floor(total_timesteps * VALIDATION_SIZE_PROPORTION))
    requested_test = int(np.floor(total_timesteps * TEST_SIZE_PROPORTION))

    idx = 0
    train_start = all_timestamps[idx] if requested_train > 0 else None
    train_end = all_timestamps[idx + requested_train - 1] if requested_train > 0 else None
    idx += requested_train

    validation_start = all_timestamps[idx] if requested_validation > 0 and idx < total_timesteps else None
    validation_end = all_timestamps[idx + requested_validation - 1] if requested_validation > 0 and idx + requested_validation <= total_timesteps else None
    idx += requested_validation

    test_start = all_timestamps[idx] if requested_test > 0 and idx < total_timesteps else None
    test_end = all_timestamps[idx + requested_test - 1] if requested_test > 0 and idx + requested_test <= total_timesteps else None

    return {
        'all_timestamps': all_timestamps,
        'total_timesteps': total_timesteps,
        'train_start': train_start,
        'train_end': train_end,
        'validation_start': validation_start,
        'validation_end': validation_end,
        'test_start': test_start,
        'test_end': test_end,
        'requested_train': requested_train,
        'requested_validation': requested_validation,
        'requested_test': requested_test,
    }


split_info = calculate_split_boundaries()
TRAIN_END_DATE = split_info['train_end'] if split_info['train_end'] is not None else AVAILABLE_DATA_START_DATE - TIMESTEP_RESOLUTION
VALIDATION_END_DATE = split_info['validation_end'] if split_info['validation_end'] is not None else TRAIN_END_DATE
TEST_END_DATE = split_info['test_end'] if split_info['test_end'] is not None else VALIDATION_END_DATE

print('Dataset split boundaries')
print(f"  Training:   {split_info['train_start']} to {split_info['train_end']}")
print(f"  Validation: {split_info['validation_start']} to {split_info['validation_end']}")
print(f"  Test:       {split_info['test_start']} to {split_info['test_end']}")
print(f'  Total available timesteps: {split_info["total_timesteps"]}')


def calculate_cyclical_features(timestamp):
    timestamp = pd.Timestamp(timestamp)
    minutes_since_midnight = timestamp.hour * 60 + timestamp.minute
    half_hours_since_midnight = minutes_since_midnight / 30
    day_of_year = timestamp.dayofyear
    days_in_year = 366 if timestamp.is_leap_year else 365
    return {
        'sin_time_of_day': np.sin(2 * np.pi * half_hours_since_midnight / 48.0),
        'cos_time_of_day': np.cos(2 * np.pi * half_hours_since_midnight / 48.0),
        'sin_day_of_year': np.sin(2 * np.pi * day_of_year / days_in_year),
        'cos_day_of_year': np.cos(2 * np.pi * day_of_year / days_in_year),
    }


def safe_to_numeric(df, columns):
    for col in columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


## Upstream Data Processing Functions

The following cells implement the standalone scripts as reusable notebook functions. They are safe to define repeatedly. They only write files when the pipeline runner is enabled.


In [ ]:
# ============================================================================
# [PART 3] Raw BOM Fixed-Width Processing: process_data.py
# ============================================================================

# Cell notes:
# - Recreates `process_data.py`: parses Bureau of Meteorology fixed-width raw station files.
# - Converts raw date/time fields into `timestamp`, coerces weather values to numeric, and writes cleaned station files.
# - Use this only when raw `RawDataC/*Data*.txt` files are available.

RAW_COL_SPECS = [
    (0, 2), (3, 9), (10, 14), (15, 17), (18, 20), (21, 23), (24, 26),
    (27, 32), (33, 34), (35, 40), (41, 42), (43, 48), (49, 50),
    (51, 54), (55, 56), (57, 62), (63, 64), (65, 66), (67, 68),
    (69, 70), (71, 72), (73, 74), (75, 76), (77, 78), (79, 80),
    (81, 87), (88, 89), (90, 92), (93, 94),
]
RAW_COL_NAMES = [
    'record_id', 'station_id', 'year', 'month', 'day', 'hour', 'minute',
    'air_temp', 'q_air_temp', 'dew_point', 'q_dew_point', 'wind_speed',
    'q_wind_speed', 'wind_dir', 'q_wind_dir', 'max_gust_speed',
    'q_max_gust_speed', 'cloud1_amt', 'q_cloud1_amt', 'cloud2_amt',
    'q_cloud2_amt', 'cloud3_amt', 'q_cloud3_amt', 'cloud4_amt',
    'q_cloud4_amt', 'msl_pressure', 'q_msl_pressure', 'aws_flag',
    'end_indicator',
]
RAW_FINAL_VALUE_COLUMNS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']
RAW_NUMERIC_COLUMNS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'max_gust_speed', 'msl_pressure', 'aws_flag']


def process_raw_weather_file(filepath, output_dir=PROCESSED_DATA_DIR):
    df = pd.read_fwf(filepath, colspecs=RAW_COL_SPECS, names=RAW_COL_NAMES, dtype=str, skiprows=1)
    if df.empty:
        print(f'Skipping empty raw file: {filepath}')
        return None

    station_ids = df['station_id'].dropna().astype(str).str.strip()
    if station_ids.empty or not station_ids.iloc[0]:
        print(f'Could not determine station ID for raw file: {filepath}')
        return None
    station_id = station_ids.iloc[0]

    df['timestamp_str'] = (
        df['year'].str.strip() + '-' + df['month'].str.strip() + '-' + df['day'].str.strip() + ' ' +
        df['hour'].str.strip() + ':' + df['minute'].str.strip()
    )
    df['timestamp'] = pd.to_datetime(df['timestamp_str'], format='%Y-%m-%d %H:%M', errors='coerce')

    for col in RAW_NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.strip(), errors='coerce')
        else:
            df[col] = np.nan

    df['data_completeness'] = df[RAW_FINAL_VALUE_COLUMNS].notna().all(axis=1) & df['timestamp'].notna()
    df['data_completeness'] = df['data_completeness'].astype(int)

    processed_df = df[['timestamp'] + RAW_FINAL_VALUE_COLUMNS + ['data_completeness']].copy()
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f'{station_id}.txt'
    processed_df.to_csv(output_path, index=False, na_rep='NaN')
    print(f'Processed raw file {filepath.name} -> {output_path}')
    return output_path


def process_raw_data_folder(raw_dir=RAW_DATA_DIR, output_dir=PROCESSED_DATA_DIR):
    raw_files = sorted(raw_dir.glob('*Data*.txt'))
    if not raw_files:
        print(f'No raw BOM files found in {raw_dir}')
        return []
    return [p for p in (process_raw_weather_file(file, output_dir) for file in raw_files) if p is not None]

print('Raw fixed-width processing functions ready')


In [ ]:
# ============================================================================
# [PART 4] Station Merging, Cropping, Linear Interpolation, U/V Conversion, and Column Dropping
# ============================================================================

# Cell notes:
# - Recreates the intermediate processing scripts: station merge, year crop, single-step interpolation, wind U/V conversion, and final column dropping.
# - These functions write intermediate station files when called manually.
# - They do not run automatically; call the specific function you need.

MERGE_WEATHER_PARAMS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'max_gust_speed', 'msl_pressure', 'aws_flag']


def merge_weather_data(file1_path, file2_path, output_path):
    df1 = pd.read_csv(file1_path, parse_dates=['timestamp'], na_values=['NaN'])
    df2 = pd.read_csv(file2_path, parse_dates=['timestamp'], na_values=['NaN'])
    for df in [df1, df2]:
        safe_to_numeric(df, [col for col in MERGE_WEATHER_PARAMS if col in df.columns])

    merged_df = pd.merge(df1, df2, on='timestamp', how='outer', suffixes=('_f1', '_f2'))
    for param in MERGE_WEATHER_PARAMS:
        f1 = f'{param}_f1'
        f2 = f'{param}_f2'
        if f1 in merged_df.columns and f2 in merged_df.columns:
            merged_df[param] = merged_df[f2].combine_first(merged_df[f1])
        elif f2 in merged_df.columns:
            merged_df[param] = merged_df[f2]
        elif f1 in merged_df.columns:
            merged_df[param] = merged_df[f1]
        else:
            merged_df[param] = np.nan

    cols_for_completeness = [col for col in MERGE_WEATHER_PARAMS if col in merged_df.columns]
    merged_df['data_completeness'] = merged_df[cols_for_completeness].notna().all(axis=1).astype(int)
    output_df = merged_df[['timestamp'] + MERGE_WEATHER_PARAMS + ['data_completeness']].sort_values('timestamp')
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_csv(output_path, index=False, na_rep='NaN')
    print(f'Merged station data saved to {output_path}')
    return output_path


def crop_files_by_year_range(input_dir, output_dir, start_year, end_year):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    start = pd.Timestamp(f'{start_year}-01-01 00:00:00')
    end = pd.Timestamp(f'{end_year}-12-31 23:59:59')
    written = []
    for file in sorted(input_dir.glob('*.txt')):
        df = pd.read_csv(file)
        if 'timestamp' not in df.columns:
            print(f'Skipping {file.name}: timestamp column missing')
            continue
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])
        df = df[(df['timestamp'] >= start) & (df['timestamp'] <= end)]
        output_path = output_dir / file.name
        df.to_csv(output_path, index=False, na_rep='NaN')
        written.append(output_path)
    print(f'Cropped {len(written)} files to {output_dir}')
    return written


def interpolate_single_step_gaps(series):
    interpolated = series.copy()
    is_na = series.isna()
    for i in range(1, len(series) - 1):
        if not is_na.iloc[i - 1] and is_na.iloc[i] and not is_na.iloc[i + 1]:
            interpolated.iloc[i] = (series.iloc[i - 1] + series.iloc[i + 1]) / 2.0
    return interpolated


def interpolate_single_step_files(input_dir=PROCESSED_DATA_DIR, output_dir=INTERPOLATED_DATA_DIR, start_year=None):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for file in sorted(input_dir.glob('*.txt')):
        df = pd.read_csv(file, parse_dates=['timestamp'], na_values=['NaN'])
        if start_year is not None:
            df = df[df['timestamp'].dt.year >= start_year].copy()
        for element in LINEAR_INTERPOLATION_ELEMENTS:
            if element in df.columns:
                df[element] = pd.to_numeric(df[element], errors='coerce')
                df[element] = interpolate_single_step_gaps(df[element])
        if 'data_completeness' in df.columns:
            cols = [el for el in LINEAR_INTERPOLATION_ELEMENTS if el in df.columns]
            df['data_completeness'] = (df[cols].notna().all(axis=1) & df['timestamp'].notna()).astype(int)
        output_path = output_dir / file.name
        df.to_csv(output_path, index=False, na_rep='NaN', float_format='%.3f')
        written.append(output_path)
    print(f'Linear interpolation wrote {len(written)} files to {output_dir}')
    return written


def convert_wind_to_uv(input_dir=INTERPOLATED_DATA_DIR, output_dir=UV_COMPONENT_DATA_DIR, start_year=None):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for file in sorted(input_dir.glob('*.txt')):
        df = pd.read_csv(file, parse_dates=['timestamp'], na_values=['NaN'])
        if start_year is not None:
            df = df[df['timestamp'].dt.year >= start_year].copy()
        if 'wind_speed' in df.columns and 'wind_dir' in df.columns:
            speed = pd.to_numeric(df['wind_speed'], errors='coerce')
            direction_rad = np.deg2rad(pd.to_numeric(df['wind_dir'], errors='coerce').astype(float))
            df['u_component'] = -speed * np.sin(direction_rad)
            df['v_component'] = speed * np.cos(direction_rad)
            mask_nan = speed.isna() | pd.to_numeric(df['wind_dir'], errors='coerce').isna()
            df.loc[mask_nan, ['u_component', 'v_component']] = np.nan
            df = df.drop(columns=['wind_speed', 'wind_dir'])
        output_path = output_dir / file.name
        df.to_csv(output_path, index=False, na_rep='NaN', float_format='%.3f')
        written.append(output_path)
    print(f'U/V conversion wrote {len(written)} files to {output_dir}')
    return written


def drop_columns_from_files(input_dir=SPLINE_INTERPOLATED_DATA_DIR, output_dir=FINAL_DATA_DIR, columns_to_drop=None):
    columns_to_drop = COLUMNS_TO_DROP_FROM_FINAL if columns_to_drop is None else columns_to_drop
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for file in sorted(input_dir.glob('*.txt')):
        df = pd.read_csv(file)
        df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
        output_path = output_dir / file.name
        df.to_csv(output_path, index=False, na_rep='NaN')
        written.append(output_path)
    print(f'Dropped columns and wrote {len(written)} files to {output_dir}')
    return written

print('Intermediate processing functions ready')


In [ ]:
# ============================================================================
# [PART 5] Spline Interpolation: interpolate.py
# ============================================================================

# Cell notes:
# - Recreates `interpolate.py`: fills short NaN gaps with local cubic spline interpolation.
# - Uses `UnivariateSpline` from SciPy with context points on both sides of each gap.
# - Only gaps up to `MAX_GAP_STEPS_FOR_SPLINE` are filled; longer outages remain missing.

def cubic_spline_interpolate_gap(series_with_gap, gap_start_iloc, gap_end_iloc, context_points, k=3, s=0):
    if not SCIPY_AVAILABLE:
        raise ImportError('Spline interpolation requires scipy. Install it with: pip install scipy')

    n = len(series_with_gap)
    before = series_with_gap.iloc[max(0, gap_start_iloc - context_points):gap_start_iloc].dropna()
    after = series_with_gap.iloc[gap_end_iloc + 1:min(n, gap_end_iloc + 1 + context_points)].dropna()
    known = pd.concat([before, after])
    if len(known) < k + 1:
        return None

    x_known = np.array([series_with_gap.index.get_loc(idx) for idx in known.index])
    y_known = known.values.astype(float)
    order = np.argsort(x_known)
    x_known = x_known[order]
    y_known = y_known[order]
    x_unique, unique_idx = np.unique(x_known, return_index=True)
    if len(x_unique) < k + 1:
        return None

    spline = UnivariateSpline(x_unique, y_known[unique_idx], k=k, s=s)
    gap_ilocs = np.arange(gap_start_iloc, gap_end_iloc + 1)
    return pd.Series(spline(gap_ilocs), index=series_with_gap.index[gap_ilocs])


def apply_spline_interpolation_to_series(series_original, series_name='series'):
    series = series_original.copy()
    is_na = series_original.isna()
    events = []
    gap_start = None

    for i in range(len(series_original)):
        if is_na.iloc[i] and gap_start is None:
            gap_start = i
        is_gap_end = gap_start is not None and ((not is_na.iloc[i]) or i == len(series_original) - 1)
        if is_gap_end:
            gap_end = i - 1 if not is_na.iloc[i] else i
            gap_len = gap_end - gap_start + 1
            if 0 < gap_len <= MAX_GAP_STEPS_FOR_SPLINE:
                filled = cubic_spline_interpolate_gap(
                    series_original,
                    gap_start,
                    gap_end,
                    SPLINE_CONTEXT_POINTS,
                    k=SPLINE_ORDER,
                    s=SPLINE_SMOOTHING_FACTOR,
                )
                if filled is not None and not filled.empty:
                    series.loc[filled.index] = filled.values
                    events.append({
                        'time_start_gap': series_original.index[gap_start],
                        'time_end_gap': series_original.index[gap_end],
                        'method': f'spline(k={SPLINE_ORDER},s={SPLINE_SMOOTHING_FACTOR},ctx={SPLINE_CONTEXT_POINTS})',
                        'len_steps': gap_len,
                        'series': series_name,
                    })
            gap_start = None
    return series, events


def run_spline_interpolation(input_dir=UV_COMPONENT_DATA_DIR, output_dir=SPLINE_INTERPOLATED_DATA_DIR, start_year=None):
    if not SCIPY_AVAILABLE:
        raise ImportError('Spline interpolation requires scipy. Install it with: pip install scipy')

    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    all_events = []
    written = []

    for file in sorted(input_dir.glob('*.txt')):
        station_id = file.stem
        df = pd.read_csv(file, parse_dates=['timestamp'], na_values=['NaN'])
        if start_year is not None:
            df = df[df['timestamp'].dt.year >= start_year].copy()
        if df.empty:
            continue
        df = df.set_index('timestamp').sort_index()
        df_to_save = df.copy()

        for element in SPLINE_INTERPOLATION_ELEMENTS:
            if element not in df.columns:
                continue
            df[element] = pd.to_numeric(df[element], errors='coerce')
            filled, events = apply_spline_interpolation_to_series(df[element], series_name=element)
            df_to_save[element] = filled
            for event in events:
                event['station_id'] = station_id
                event['element'] = element
            all_events.extend(events)

        if 'u_component' in df_to_save.columns and 'v_component' in df_to_save.columns:
            df_to_save['wind_speed_recalc'] = np.sqrt(df_to_save['u_component'] ** 2 + df_to_save['v_component'] ** 2)
            direction_rad = np.arctan2(-df_to_save['u_component'], -df_to_save['v_component'])
            df_to_save['wind_dir_recalc'] = (np.rad2deg(direction_rad) + 360) % 360
            df_to_save['wind_dir_recalc'] = df_to_save['wind_dir_recalc'].where(df_to_save['wind_speed_recalc'] > 0.01, np.nan)

        if 'data_completeness' in df_to_save.columns:
            cols = [col for col in ['air_temp', 'dew_point', 'msl_pressure', 'wind_speed_recalc', 'wind_dir_recalc'] if col in df_to_save.columns]
            if cols:
                df_to_save['data_completeness'] = df_to_save[cols].notna().all(axis=1).astype(int)

        output_path = output_dir / file.name
        df_to_save.reset_index().to_csv(output_path, index=False, na_rep='NaN', float_format='%.3f')
        written.append(output_path)

    print(f'Spline interpolation wrote {len(written)} files to {output_dir}')
    print(f'Recorded {len(all_events)} interpolation events')
    return written, pd.DataFrame(all_events)

print('Spline interpolation functions ready')


In [ ]:
# ============================================================================
# [PART 6] Diagnostics: Missing Timestamps, Availability, and Outage Lengths
# ============================================================================

# Cell notes:
# - Defines diagnostic helpers from the timestamp, availability, and outage scripts.
# - These helpers summarise missing timestamps, monthly element availability, and outage duration distributions.
# - They are intended for data-quality inspection before model training.

OUTAGE_BINS_CONFIG = [
    (1, 2, '30 min'), (2, 3, '1 hr'), (3, 5, '1.5-2 hr'), (5, 13, '2-6 hr'),
    (13, 49, '6-24 hr'), (49, 337, '1-7 day'), (337, 1441, '7day-1mo'),
    (1441, 17521, '1mo-1yr'), (17521, float('inf'), '>1 yr'),
]
OUTAGE_ELEMENTS = ['air_temp', 'dew_point', 'wind_speed', 'wind_dir', 'msl_pressure']


def find_missing_timestamps(file_path, start=AVAILABLE_DATA_START_DATE, end=AVAILABLE_DATA_END_DATE):
    df = pd.read_csv(file_path, parse_dates=['timestamp'], na_values=['NaN'])
    expected = pd.date_range(start=start, end=end, freq=TIMESTEP_RESOLUTION)
    present = pd.DatetimeIndex(df['timestamp'].dropna().unique())
    missing = expected.difference(present)
    extra = present.difference(expected)
    return {'file': Path(file_path).name, 'missing': missing, 'extra': extra}


def calculate_monthly_availability(folder_path=PROCESSED_DATA_DIR, elements=None):
    elements = OUTAGE_ELEMENTS if elements is None else elements
    records = []
    for file in sorted(Path(folder_path).glob('*.txt')):
        df = pd.read_csv(file, parse_dates=['timestamp'], na_values=['NaN'])
        df = df[df['timestamp'].dt.minute == 0].copy()
        if df.empty:
            continue
        df['year_month'] = df['timestamp'].dt.to_period('M')
        for element in elements:
            if element not in df.columns:
                continue
            summary = df.groupby('year_month').agg(
                total_possible=('timestamp', 'count'),
                available=(element, lambda s: s.notna().sum()),
            ).reset_index()
            summary['station'] = file.stem
            summary['element'] = element
            summary['percentage_available'] = summary['available'] / summary['total_possible'] * 100
            records.append(summary)
    if not records:
        return pd.DataFrame()
    return pd.concat(records, ignore_index=True)


def find_outage_durations(series):
    outages = []
    current = 0
    for is_missing in series.isna():
        if is_missing:
            current += 1
        elif current > 0:
            outages.append(current)
            current = 0
    if current > 0:
        outages.append(current)
    return outages


def categorise_outages(outage_durations_steps, bins_def=OUTAGE_BINS_CONFIG):
    counts = {label: 0 for _, _, label in bins_def}
    for duration in outage_durations_steps:
        for lower, upper, label in bins_def:
            if lower <= duration < upper:
                counts[label] += 1
                break
    return counts


def summarise_outages(folder_path=FINAL_DATA_DIR, elements=None, start_year=None):
    elements = OUTAGE_ELEMENTS if elements is None else elements
    records = []
    for file in sorted(Path(folder_path).glob('*.txt')):
        df = pd.read_csv(file, parse_dates=['timestamp'], na_values=['NaN'])
        if start_year is not None:
            df = df[df['timestamp'].dt.year >= start_year].copy()
        if df.empty:
            continue
        for element in elements:
            if element not in df.columns:
                continue
            outages = find_outage_durations(df[element])
            counts = categorise_outages(outages)
            for duration_label, count in counts.items():
                records.append({
                    'station': file.stem,
                    'element': element,
                    'duration_category': duration_label,
                    'outage_count': count,
                    'uptime_pct': df[element].notna().mean() * 100,
                })
    return pd.DataFrame(records)

print('Diagnostic functions ready')


In [ ]:
# ============================================================================
# [PART 7] Optional Upstream Pipeline and Diagnostics Runners
# ============================================================================

# Cell notes:
# - Provides manual runner functions for upstream processing and diagnostics.
# - `run_upstream_data_pipeline()` rebuilds intermediate station files and may overwrite folders.
# - `run_data_diagnostics()` writes Parquet summaries for availability and outage metrics.

def run_upstream_data_pipeline():
    """Run the full upstream station-data pipeline and write intermediate files."""
    print('Running upstream data pipeline. This may overwrite intermediate output folders.')
    if RAW_DATA_DIR.exists():
        process_raw_data_folder(RAW_DATA_DIR, PROCESSED_DATA_DIR)
    else:
        print(f'Skipping raw processing: {RAW_DATA_DIR} does not exist')

    if PROCESSED_DATA_DIR.exists():
        interpolate_single_step_files(PROCESSED_DATA_DIR, INTERPOLATED_DATA_DIR)
    else:
        print(f'Skipping linear interpolation: {PROCESSED_DATA_DIR} does not exist')

    if INTERPOLATED_DATA_DIR.exists():
        convert_wind_to_uv(INTERPOLATED_DATA_DIR, UV_COMPONENT_DATA_DIR)
    else:
        print(f'Skipping U/V conversion: {INTERPOLATED_DATA_DIR} does not exist')

    if UV_COMPONENT_DATA_DIR.exists():
        run_spline_interpolation(UV_COMPONENT_DATA_DIR, SPLINE_INTERPOLATED_DATA_DIR)
    else:
        print(f'Skipping spline interpolation: {UV_COMPONENT_DATA_DIR} does not exist')

    if SPLINE_INTERPOLATED_DATA_DIR.exists():
        drop_columns_from_files(SPLINE_INTERPOLATED_DATA_DIR, FINAL_DATA_DIR)
    else:
        print(f'Skipping final column dropping: {SPLINE_INTERPOLATED_DATA_DIR} does not exist')


def run_data_diagnostics():
    """Create Parquet summaries for monthly availability and outage lengths."""
    require_pyarrow()
    print('Running diagnostics')

    availability_df = calculate_monthly_availability(PROCESSED_DATA_DIR)
    if not availability_df.empty:
        availability_path = DIAGNOSTICS_DIR / 'monthly_availability.parquet'
        availability_df.to_parquet(availability_path, index=False)
        print(f'Saved monthly availability summary to {availability_path}')
    else:
        print('Monthly availability summary is empty')

    outage_df = summarise_outages(FINAL_DATA_DIR, start_year=2005)
    if not outage_df.empty:
        outage_path = DIAGNOSTICS_DIR / 'outage_summary.parquet'
        outage_df.to_parquet(outage_path, index=False)
        print(f'Saved outage summary to {outage_path}')
    else:
        print('Outage summary is empty')

print('Upstream pipeline and diagnostic runner functions ready')
# Run manually when needed:
# run_upstream_data_pipeline()
# run_data_diagnostics()


## FinalData to Aligned Weather Parquet

This is the hand-off point requested for the notebook: weather station files are prepared into one aligned Parquet file, and every downstream dataset step reads from that Parquet-backed DataFrame.


In [ ]:
# ============================================================================
# [PART 8] FinalData -> aligned_weather_data.parquet
# ============================================================================

# Cell notes:
# - Converts prepared `FinalData/*.txt` station files into one aligned weather-station Parquet file.
# - Reindexes every station to a full 30-minute timeline from 2000 to 2024.
# - Loads the Parquet back into `combined_df`, which is the only input used by downstream sample generation.

def weather_parquet_columns(stations):
    columns = ['timestamp']
    for station in stations:
        for feature in get_station_features(station):
            columns.append(f'{station}_{feature}')
    return columns


def prepare_weather_data_to_parquet(data_folder=FINAL_DATA_DIR, stations=None, output_parquet=ALIGNED_DATA_PARQUET):
    require_pyarrow()
    stations = STATIONS if stations is None else stations
    data_folder = Path(data_folder)
    output_parquet = Path(output_parquet)

    date_range = pd.date_range(
        start=AVAILABLE_DATA_START_DATE,
        end=AVAILABLE_DATA_END_DATE,
        freq=TIMESTEP_RESOLUTION,
    )

    flattened_data = {'timestamp': date_range}
    for station in stations:
        file_path = data_folder / f'{station}.txt'
        features = get_station_features(station)
        if not file_path.exists():
            print(f'{station}: file missing; creating NaN columns')
            df_aligned = pd.DataFrame(index=date_range, columns=features, dtype='float64')
        else:
            df = pd.read_csv(file_path, parse_dates=['timestamp'], na_values=['NaN'])
            df = df.set_index('timestamp').sort_index()
            for col in ALL_FILE_FEATURES:
                if col not in df.columns:
                    df[col] = np.nan
            if station in STATIONS_NO_PRESSURE and 'msl_pressure' in df.columns:
                df = df.drop(columns=['msl_pressure'])
            df = df.reindex(columns=features)
            df_aligned = df.reindex(date_range)
        for feature in features:
            flattened_data[f'{station}_{feature}'] = pd.to_numeric(df_aligned[feature], errors='coerce').values

    df_flat = pd.DataFrame(flattened_data, columns=weather_parquet_columns(stations))
    output_parquet.parent.mkdir(parents=True, exist_ok=True)
    df_flat.to_parquet(output_parquet, engine='pyarrow', compression='snappy', index=False)
    print(f'Saved aligned weather Parquet: {output_parquet}')
    print(f'Weather Parquet shape: {df_flat.shape}')
    return output_parquet


def load_weather_data_from_parquet(parquet_path=ALIGNED_DATA_PARQUET, stations=None):
    require_pyarrow()
    stations = STATIONS if stations is None else stations
    parquet_path = Path(parquet_path)
    df_parquet = pd.read_parquet(parquet_path, engine='pyarrow')
    missing = [col for col in weather_parquet_columns(stations) if col not in df_parquet.columns]
    if missing:
        raise ValueError(f'Aligned weather Parquet is missing expected columns: {missing[:5]}')

    combined_df = pd.DataFrame(index=pd.to_datetime(df_parquet['timestamp']))
    ordered_columns = []
    for station in stations:
        for feature in get_station_features(station):
            tuple_col = (station, feature)
            combined_df[tuple_col] = df_parquet[f'{station}_{feature}'].values
            ordered_columns.append(tuple_col)
    combined_df = combined_df[ordered_columns]
    combined_df.columns = pd.MultiIndex.from_tuples(combined_df.columns, names=['station', 'feature'])
    return combined_df


prepare_weather_data_to_parquet(FINAL_DATA_DIR, STATIONS, ALIGNED_DATA_PARQUET)
combined_df = load_weather_data_from_parquet(ALIGNED_DATA_PARQUET, STATIONS)
print(f'Loaded combined_df from Parquet: {combined_df.shape}')


In [ ]:
# ============================================================================
# [PART 9] ML Feature Engineering Matching preprocess_data.py
# ============================================================================

# Cell notes:
# - Defines the feature-engineering logic that matches `preprocess_data.py`.
# - Builds semantic feature names: station/feature lags, pressure gradients, pressure trends, time derivatives, BELL V-wind derivatives, and cyclical time features.
# - Defines target names such as `YSSY_u_forecast_t_plus_1` through `YSSY_v_forecast_t_plus_48`.

def get_val(window_df, station, feature, time_idx):
    if not (0 <= time_idx < len(window_df)):
        return np.nan
    if (station, feature) not in window_df.columns:
        return np.nan
    return window_df.iloc[time_idx][(station, feature)]


def validate_sample_windows(input_window_df, output_window_df):
    if input_window_df.shape[0] != NUM_LOOKBACK_STEPS:
        return False, 'input window wrong length'
    if output_window_df.shape[0] != NUM_FORECAST_STEPS:
        return False, 'output window wrong length'

    for station in STATIONS:
        for feature in get_station_features(station):
            if (station, feature) not in input_window_df.columns:
                return False, f'missing input column {(station, feature)}'
            if input_window_df[(station, feature)].isna().any():
                return False, f'NaN input for {station}-{feature}'

    for feature in ['u_component', 'v_component']:
        if (TARGET_STATION, feature) not in output_window_df.columns:
            return False, f'missing target column {(TARGET_STATION, feature)}'
        if output_window_df[(TARGET_STATION, feature)].isna().any():
            return False, f'NaN output for {TARGET_STATION}-{feature}'

    return True, ''


def build_feature_dict(input_window_df, current_t_timestamp):
    feature_dict = {}
    idx_t = NUM_LOOKBACK_STEPS - 1
    steps_per_3hr = int(3 * 60 / TIMESTEP_MINUTES)
    num_3hr_periods = int(LOOKBACK_WINDOW_HOURS / 3)

    idx_t_minus_6hr = idx_t - int(6 * 60 / TIMESTEP_MINUTES)
    idx_t_minus_12hr = idx_t - int(12 * 60 / TIMESTEP_MINUTES)
    idx_t_minus_24hr = 0

    # A. Raw lagged features. The target station keeps every 30-minute step;
    # other stations keep the most recent 6 hours at 30-minute resolution and older data at hourly resolution.
    for station in STATIONS:
        for feature_name in get_station_features(station):
            series_values = input_window_df[(station, feature_name)].values
            if station == TARGET_STATION:
                lag_range = range(NUM_LOOKBACK_STEPS)
            else:
                lag_range = list(range(12)) + list(range(12, NUM_LOOKBACK_STEPS, 2))
            for lag in lag_range:
                step_idx = (NUM_LOOKBACK_STEPS - 1) - lag
                feature_dict[f'{station}_{feature_name}_lag{lag}'] = series_values[step_idx]

    # B. Cyclical time features.
    feature_dict.update(calculate_cyclical_features(current_t_timestamp))

    # C. Pressure gradients and pressure-gradient trends.
    pg_pairs = [
        ('YBTH', 'YSSY', 'PG_YBTH_YSSY'),
        ('YSSY', 'YCNK', 'PG_YSSY_YCNK'),
        ('YSSY', 'YSNW', 'PG_YSSY_YSNW'),
    ]
    pg_time_indices = {
        't': idx_t,
        't_minus_6hr': idx_t_minus_6hr,
        't_minus_12hr': idx_t_minus_12hr,
        't_minus_24hr': idx_t_minus_24hr,
    }
    for s1, s2, pair_label in pg_pairs:
        gradients = {}
        for time_label, time_idx in pg_time_indices.items():
            gradient = get_val(input_window_df, s1, 'msl_pressure', time_idx) - get_val(input_window_df, s2, 'msl_pressure', time_idx)
            feature_dict[f'{pair_label}_{time_label}'] = gradient
            gradients[time_label] = gradient
        feature_dict[f'{pair_label}_trend_t_vs_t_minus_6hr'] = gradients['t'] - gradients['t_minus_6hr']
        feature_dict[f'{pair_label}_trend_t_vs_t_minus_12hr'] = gradients['t'] - gradients['t_minus_12hr']

    # D. Time derivatives for temperature, dewpoint, and pressure.
    for station in STATIONS:
        for param in ['air_temp', 'dew_point', 'msl_pressure']:
            if param == 'msl_pressure' and station in STATIONS_NO_PRESSURE:
                continue
            for period_num in range(num_3hr_periods):
                end_offset_steps = period_num * steps_per_3hr
                start_offset_steps = (period_num + 1) * steps_per_3hr
                idx_period_end = idx_t - end_offset_steps
                idx_period_start = 0 if start_offset_steps == NUM_LOOKBACK_STEPS else idx_t - start_offset_steps
                if not (0 <= idx_period_start < idx_period_end < NUM_LOOKBACK_STEPS):
                    continue
                end_hr = end_offset_steps // (60 // TIMESTEP_MINUTES)
                start_hr = start_offset_steps // (60 // TIMESTEP_MINUTES)
                feature_dict[f'{station}_{param}_deriv_t_minus_{end_hr}hr_vs_t_minus_{start_hr}hr'] = (
                    get_val(input_window_df, station, param, idx_period_end) -
                    get_val(input_window_df, station, param, idx_period_start)
                )
            feature_dict[f'{station}_{param}_deriv_t_vs_t_minus_6hr'] = (
                get_val(input_window_df, station, param, idx_t) - get_val(input_window_df, station, param, idx_t_minus_6hr)
            )
            feature_dict[f'{station}_{param}_deriv_t_vs_t_minus_12hr'] = (
                get_val(input_window_df, station, param, idx_t) - get_val(input_window_df, station, param, idx_t_minus_12hr)
            )

    # E. BELL V-component derivatives.
    station_bell = 'BELL'
    for step_offset in range(12):
        idx_current = idx_t - step_offset
        idx_30min = idx_current - int(0.5 * 60 / TIMESTEP_MINUTES)
        idx_60min = idx_current - int(1.0 * 60 / TIMESTEP_MINUTES)
        eval_time_label_hr = f'{step_offset * 0.5:.1f}'
        if idx_30min >= 0:
            feature_dict[f'{station_bell}_v_deriv_eval_at_t_minus_{eval_time_label_hr}hr_lag_0.5hr'] = (
                get_val(input_window_df, station_bell, 'v_component', idx_current) -
                get_val(input_window_df, station_bell, 'v_component', idx_30min)
            )
        if idx_60min >= 0:
            feature_dict[f'{station_bell}_v_deriv_eval_at_t_minus_{eval_time_label_hr}hr_lag_1.0hr'] = (
                get_val(input_window_df, station_bell, 'v_component', idx_current) -
                get_val(input_window_df, station_bell, 'v_component', idx_60min)
            )

    return feature_dict


def build_target_list(output_window_df):
    target_values = []
    for step in range(NUM_FORECAST_STEPS):
        target_values.append(output_window_df[(TARGET_STATION, 'u_component')].iloc[step])
        target_values.append(output_window_df[(TARGET_STATION, 'v_component')].iloc[step])
    return target_values


TARGET_FEATURE_ORDER_Y = []
for step in range(1, NUM_FORECAST_STEPS + 1):
    TARGET_FEATURE_ORDER_Y.append(f'{TARGET_STATION}_u_forecast_t_plus_{step}')
    TARGET_FEATURE_ORDER_Y.append(f'{TARGET_STATION}_v_forecast_t_plus_{step}')

print('Feature engineering functions ready')
print(f'Target columns: {len(TARGET_FEATURE_ORDER_Y)}')


In [ ]:
# ============================================================================
# [PART 10] Generate Semantic ML Dataset from Parquet-backed combined_df
# ============================================================================

# Cell notes:
# - Generates ML samples from `combined_df` using a 24-hour input window and 24-hour forecast horizon.
# - Skips samples with missing input features or missing YSSY U/V target values.
# - Produces `full_dataset_df`, with `timestamp_t`, semantic feature columns, and semantic target columns.

all_samples_X = []
all_samples_y = []
all_sample_timestamps_t = []
FEATURE_ORDER_X = []

skipped_nan_input = 0
skipped_nan_output = 0
skipped_other = 0
included_samples = 0
skip_reason_counts = {}

start_loop_idx = NUM_LOOKBACK_STEPS - 1
# Match preprocess_data.py: only generate anchors that can produce a complete
# forecast window within the configured train/validation/test period.
last_loaded_idx = combined_df.index.searchsorted(TEST_END_DATE, side='right') - 1
max_t_idx = min(len(combined_df) - 1 - NUM_FORECAST_STEPS, last_loaded_idx - NUM_FORECAST_STEPS)
print(f'Generating samples for t index {start_loop_idx} to {max_t_idx}')
print(f'  Last configured data timestamp for samples: {TEST_END_DATE}')

for i in range(start_loop_idx, max_t_idx + 1):
    current_t_timestamp = combined_df.index[i]
    input_window_df = combined_df.iloc[i - NUM_LOOKBACK_STEPS + 1:i + 1]
    output_window_df = combined_df.iloc[i + 1:i + NUM_FORECAST_STEPS + 1]

    is_valid, reason = validate_sample_windows(input_window_df, output_window_df)
    if not is_valid:
        skip_reason_counts[reason] = skip_reason_counts.get(reason, 0) + 1
        if reason.startswith('NaN input'):
            skipped_nan_input += 1
        elif reason.startswith('NaN output'):
            skipped_nan_output += 1
        else:
            skipped_other += 1
        continue

    feature_dict = build_feature_dict(input_window_df, current_t_timestamp)
    if not FEATURE_ORDER_X:
        FEATURE_ORDER_X = sorted(feature_dict.keys())
        print(f'Established FEATURE_ORDER_X with {len(FEATURE_ORDER_X)} features')

    all_samples_X.append([feature_dict[col] for col in FEATURE_ORDER_X])
    all_samples_y.append(build_target_list(output_window_df))
    all_sample_timestamps_t.append(current_t_timestamp)
    included_samples += 1

    if included_samples == 1 or included_samples % 1000 == 0:
        print(f'  Generated {included_samples} valid samples through {current_t_timestamp}')

if included_samples == 0:
    print('No valid samples were generated.')
    print('Top skip reasons:')
    for reason, count in sorted(skip_reason_counts.items(), key=lambda item: item[1], reverse=True)[:20]:
        print(f'  {count:8d}  {reason}')
    raise ValueError(
        'No valid samples were generated. Rebuild aligned_weather_data.parquet from current FinalData '
        'and inspect the skip reasons above.'
    )

full_dataset_df = pd.concat(
    [
        pd.Series(pd.to_datetime(all_sample_timestamps_t), name='timestamp_t').reset_index(drop=True),
        pd.DataFrame(all_samples_X, columns=FEATURE_ORDER_X).reset_index(drop=True),
        pd.DataFrame(all_samples_y, columns=TARGET_FEATURE_ORDER_Y).reset_index(drop=True),
    ],
    axis=1,
)

print('Sample generation complete')
print(f'  Full dataset shape: {full_dataset_df.shape}')
print(f'  Skipped NaN input: {skipped_nan_input}')
print(f'  Skipped NaN output: {skipped_nan_output}')
print(f'  Skipped other: {skipped_other}')


In [ ]:
# ============================================================================
# [PART 11] Chronological Split and Dataset Save
# ============================================================================

# Cell notes:
# - Splits `full_dataset_df` chronologically into training, validation, and test datasets.
# - Saves each split as Parquet, plus feature-order and target-order metadata.
# - These Parquet files are what the training cells load when you want to skip preprocessing.

require_pyarrow()

full_dataset_df['timestamp_t'] = pd.to_datetime(full_dataset_df['timestamp_t'])
train_df = full_dataset_df[full_dataset_df['timestamp_t'] <= TRAIN_END_DATE].copy()
validation_df = full_dataset_df[
    (full_dataset_df['timestamp_t'] > TRAIN_END_DATE) &
    (full_dataset_df['timestamp_t'] <= VALIDATION_END_DATE)
].copy()
test_df = full_dataset_df[
    (full_dataset_df['timestamp_t'] > VALIDATION_END_DATE) &
    (full_dataset_df['timestamp_t'] <= TEST_END_DATE)
].copy()

print('Dataset split complete')
print(f'  Training:   {train_df.shape}')
print(f'  Validation: {validation_df.shape}')
print(f'  Test:       {test_df.shape}')

DATASET_FRAMES = {
    'training': train_df,
    'validation': validation_df,
    'test': test_df,
}

for name, df in DATASET_FRAMES.items():
    output_path = OUTPUT_DIR / f'{name}_dataset.parquet'
    df.to_parquet(output_path, engine='pyarrow', compression='snappy', index=False)
    print(f'Saved {name} dataset to {output_path}')

feature_metadata = pd.DataFrame({'feature_name': FEATURE_ORDER_X})
target_metadata = pd.DataFrame({'target_name': TARGET_FEATURE_ORDER_Y})
feature_metadata.to_parquet(OUTPUT_DIR / 'feature_order.parquet', engine='pyarrow', index=False)
target_metadata.to_parquet(OUTPUT_DIR / 'target_order.parquet', engine='pyarrow', index=False)
print('Saved feature and target order metadata')


In [ ]:
# ============================================================================
# [PART 12] Dataset Quality Checks and Visualisation
# ============================================================================

# Cell notes:
# - Computes basic dataset quality checks and visualises dataset sizes, temporal coverage, feature distribution, and target distribution.
# - Saves `dataset_statistics.png` in the output folder.
# - This is a sanity check before starting expensive model training.

feature_cols = FEATURE_ORDER_X
target_cols = TARGET_FEATURE_ORDER_Y

X_train = train_df[feature_cols]
y_train = train_df[target_cols]
X_validation = validation_df[feature_cols]
y_validation = validation_df[target_cols]
X_test = test_df[feature_cols]
y_test = test_df[target_cols]

ts_train = pd.to_datetime(train_df['timestamp_t'])
ts_validation = pd.to_datetime(validation_df['timestamp_t'])
ts_test = pd.to_datetime(test_df['timestamp_t'])

print('Dataset matrices ready')
print(f'  X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'  X_validation: {X_validation.shape}, y_validation: {y_validation.shape}')
print(f'  X_test: {X_test.shape}, y_test: {y_test.shape}')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].bar(['Training', 'Validation', 'Test'], [len(train_df), len(validation_df), len(test_df)], color=['#2b6cb0', '#c53030', '#2f855a'])
axes[0, 0].set_title('Dataset Sizes')
axes[0, 0].set_ylabel('Samples')
axes[0, 0].grid(axis='y', alpha=0.3)

if len(ts_train):
    axes[0, 1].scatter(ts_train, np.ones(len(ts_train)), s=8, label='Training', color='#2b6cb0')
if len(ts_validation):
    axes[0, 1].scatter(ts_validation, np.ones(len(ts_validation)) * 1.1, s=8, label='Validation', color='#c53030')
if len(ts_test):
    axes[0, 1].scatter(ts_test, np.ones(len(ts_test)) * 1.2, s=8, label='Test', color='#2f855a')
axes[0, 1].set_title('Temporal Distribution')
axes[0, 1].set_ylim(0.9, 1.3)
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

all_feature_values = full_dataset_df[feature_cols].to_numpy(dtype=float).ravel()
all_target_values = full_dataset_df[target_cols].to_numpy(dtype=float).ravel()
axes[1, 0].hist(all_feature_values[np.isfinite(all_feature_values)], bins=50, color='#2b6cb0', alpha=0.75)
axes[1, 0].set_title('Feature Distribution')
axes[1, 0].grid(axis='y', alpha=0.3)

axes[1, 1].hist(all_target_values[np.isfinite(all_target_values)], bins=50, color='#c53030', alpha=0.75)
axes[1, 1].set_title('Target Distribution')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plot_path = OUTPUT_DIR / 'dataset_statistics.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved dataset statistics plot to {plot_path}')


In [ ]:
# ============================================================================
# [PART 13] Load Saved Parquet Datasets
# ============================================================================

# Cell notes:
# - Defines a loader for already prepared Parquet ML datasets.
# - Use this when you want to skip preprocessing and go straight to training.
# - The returned dictionary contains DataFrames, X/y matrices, timestamps, feature column names, and target column names.

def load_ml_datasets_from_parquet(output_dir=OUTPUT_DIR):
    require_pyarrow()
    output_dir = Path(output_dir)
    datasets = {}
    for name in ['training', 'validation', 'test']:
        path = output_dir / f'{name}_dataset.parquet'
        if not path.exists():
            print(f'{name}: {path} not found')
            continue
        df = pd.read_parquet(path, engine='pyarrow')
        feature_cols_loaded = [col for col in df.columns if col not in ['timestamp_t'] and not col.startswith(f'{TARGET_STATION}_u_forecast_t_plus_') and not col.startswith(f'{TARGET_STATION}_v_forecast_t_plus_')]
        target_cols_loaded = [col for col in df.columns if col.startswith(f'{TARGET_STATION}_u_forecast_t_plus_') or col.startswith(f'{TARGET_STATION}_v_forecast_t_plus_')]
        target_cols_loaded = sorted(target_cols_loaded, key=lambda col: (int(col.split('_t_plus_')[-1]), col.split('_')[1]))
        datasets[name] = {
            'dataframe': df,
            'X': df[feature_cols_loaded],
            'y': df[target_cols_loaded],
            'timestamps': pd.to_datetime(df['timestamp_t']),
            'feature_cols': feature_cols_loaded,
            'target_cols': target_cols_loaded,
        }
        print(f'{name}: X {datasets[name]["X"].shape}, y {datasets[name]["y"].shape}')
    return datasets

print('Parquet dataset loader ready')


## Optional LightGBM Modelling

These cells adapt the standalone LightGBM scripts to the Parquet datasets produced above. They define reusable functions for quick multi-output training, single-target tuning, final individual models, model evaluation, and forecast plots.

The later execution cells load prepared Parquet data, train the quick multi-output model, compare validation and test metrics, and generate visualisations.


In [ ]:
# ============================================================================
# [PART 14] LightGBM Configuration and Dataset Helpers
# ============================================================================

# Cell notes:
# - Sets up LightGBM output folders and imports optional modelling dependencies.
# - Defines shared helpers for identifying feature columns and target columns from the prepared Parquet datasets.
# - Defines default model parameters used by the quick multi-output baseline and individual target models.

from pathlib import Path
import json

try:
    import joblib
    JOBLIB_AVAILABLE = True
except ImportError:
    JOBLIB_AVAILABLE = False

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

try:
    import optuna
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False

try:
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False

LIGHTGBM_OUTPUT_DIR = BASE_DATA_DIR / 'LightGBMNotebookOutputs'
LIGHTGBM_MODEL_DIR = LIGHTGBM_OUTPUT_DIR / 'models'
LIGHTGBM_PLOT_DIR = LIGHTGBM_OUTPUT_DIR / 'plots'
LIGHTGBM_METRICS_DIR = LIGHTGBM_OUTPUT_DIR / 'metrics'
for folder in [LIGHTGBM_OUTPUT_DIR, LIGHTGBM_MODEL_DIR, LIGHTGBM_PLOT_DIR, LIGHTGBM_METRICS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

LIGHTGBM_TARGET_PREFIXES = (
    f'{TARGET_STATION}_u_forecast_t_plus_',
    f'{TARGET_STATION}_v_forecast_t_plus_',
)

# LightGBM parameter notes:
# - `objective=regression_l2` trains by minimising squared error.
# - `metric=l2` reports mean squared error during training.
# - `n_estimators` is the maximum number of boosting trees.
# - `learning_rate` controls how much each tree contributes; smaller values are slower but often more stable.
# - `num_leaves` and `max_depth` control tree complexity.
# - `min_child_samples` prevents leaves with too few samples and helps reduce overfitting.
# - `subsample` samples rows per tree; `colsample_bytree` samples features per tree.
# - `reg_alpha` and `reg_lambda` are L1/L2 regularisation strengths.
# - `n_jobs=-1` uses all available CPU cores.
DEFAULT_QUICK_LGBM_PARAMS = {
    'objective': 'regression_l2',
    'metric': 'l2',
    'n_estimators': 100,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.0,
    'reg_lambda': 0.0,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}

DEFAULT_FINAL_INDIVIDUAL_PARAMS = {
    'learning_rate': 0.01,
    'num_leaves': 300,
    'max_depth': 40,
    'min_child_samples': 100,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,
    'reg_lambda': 0.1,
    'n_estimators': 4000,
    'objective': 'regression_l2',
    'metric': 'l2',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
}


def require_lightgbm_stack(require_optuna=False):
    missing = []
    if not LIGHTGBM_AVAILABLE:
        missing.append('lightgbm')
    if not SKLEARN_AVAILABLE:
        missing.append('scikit-learn')
    if not JOBLIB_AVAILABLE:
        missing.append('joblib')
    if require_optuna and not OPTUNA_AVAILABLE:
        missing.append('optuna')
    if missing:
        raise ImportError('Install missing LightGBM dependencies: ' + ', '.join(missing))


def get_lightgbm_target_cols(df):
    target_cols = [col for col in df.columns if col.startswith(LIGHTGBM_TARGET_PREFIXES)]
    return sorted(target_cols, key=lambda col: (int(col.split('_t_plus_')[-1]), col.split('_')[1]))


def get_lightgbm_feature_cols(df, target_cols=None):
    target_cols = get_lightgbm_target_cols(df) if target_cols is None else target_cols
    return [col for col in df.columns if col != 'timestamp_t' and col not in target_cols]


def split_lightgbm_xy(df):
    target_cols = get_lightgbm_target_cols(df)
    feature_cols = get_lightgbm_feature_cols(df, target_cols)
    X = df[feature_cols].copy()
    y = df[target_cols].copy()
    timestamps = pd.to_datetime(df['timestamp_t']) if 'timestamp_t' in df.columns else None
    return X, y, timestamps, feature_cols, target_cols


def save_joblib_artifact(obj, path):
    require_lightgbm_stack()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(obj, path)
    print(f'Saved artifact to {path}')
    return path


def load_joblib_artifact(path):
    require_lightgbm_stack()
    path = Path(path)
    obj = joblib.load(path)
    print(f'Loaded artifact from {path}')
    return obj

print('LightGBM helper functions ready')
print(f'LightGBM available: {LIGHTGBM_AVAILABLE}')
print(f'Optuna available: {OPTUNA_AVAILABLE}')


In [ ]:
# ============================================================================
# [PART 15] Quick Multi-output LightGBM Baseline: YSSY_winds_24hr.py / YSSY_LightGMB.py
# ============================================================================

# Cell notes:
# - Defines the quick baseline model from `YSSY_winds_24hr.py` / `YSSY_LightGMB.py`.
# - Model type: `MultiOutputRegressor(LGBMRegressor)`, one LightGBM regressor per target output.
# - Also defines evaluation and RMSE plotting functions for validation/test comparisons.


def train_quick_multioutput_lightgbm(train_df, params=None, subsample_frac=1.0, model_output_path=None):
    require_lightgbm_stack()
    params = DEFAULT_QUICK_LGBM_PARAMS.copy() if params is None else params.copy()
    X_train, y_train, _, feature_cols, target_cols = split_lightgbm_xy(train_df)

    if 0 < subsample_frac < 1.0:
        X_train = X_train.sample(frac=subsample_frac, random_state=42)
        y_train = y_train.loc[X_train.index]

    base_model = lgb.LGBMRegressor(**params)
    model = MultiOutputRegressor(base_model, n_jobs=-1)
    print(f'Training multi-output LightGBM on X={X_train.shape}, y={y_train.shape}')
    model.fit(X_train, y_train)

    if model_output_path is None:
        model_output_path = LIGHTGBM_MODEL_DIR / 'yssy_multi_output_model_24hr_quick.joblib'
    save_joblib_artifact(model, model_output_path)
    return model, feature_cols, target_cols


def evaluate_multioutput_lightgbm(model, test_df, feature_cols=None, target_cols=None, output_dir=LIGHTGBM_METRICS_DIR):
    require_lightgbm_stack()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    inferred_targets = get_lightgbm_target_cols(test_df)
    target_cols = inferred_targets if target_cols is None else list(target_cols)
    feature_cols = get_lightgbm_feature_cols(test_df, target_cols) if feature_cols is None else list(feature_cols)

    X_test = test_df[feature_cols]
    y_test = test_df[target_cols]
    preds = model.predict(X_test)

    overall_mae = mean_absolute_error(y_test, preds)
    overall_mse = mean_squared_error(y_test, preds)
    print(f'Overall MAE: {overall_mae:.4f}')
    print(f'Overall MSE: {overall_mse:.4f}')

    rows = []
    for step in range(1, NUM_FORECAST_STEPS + 1):
        u_col = f'{TARGET_STATION}_u_forecast_t_plus_{step}'
        v_col = f'{TARGET_STATION}_v_forecast_t_plus_{step}'
        if u_col not in target_cols or v_col not in target_cols:
            continue
        u_idx = target_cols.index(u_col)
        v_idx = target_cols.index(v_col)
        mae_u = mean_absolute_error(y_test[u_col], preds[:, u_idx])
        mse_u = mean_squared_error(y_test[u_col], preds[:, u_idx])
        mae_v = mean_absolute_error(y_test[v_col], preds[:, v_idx])
        mse_v = mean_squared_error(y_test[v_col], preds[:, v_idx])
        rows.append({
            'lead_step': step,
            'lead_hours': step * TIMESTEP_MINUTES / 60,
            'MAE_u': mae_u,
            'MSE_u': mse_u,
            'RMSE_u': np.sqrt(mse_u),
            'MAE_v': mae_v,
            'MSE_v': mse_v,
            'RMSE_v': np.sqrt(mse_v),
            'Avg_MAE': np.mean([mae_u, mae_v]),
            'Avg_MSE': np.mean([mse_u, mse_v]),
            'Avg_RMSE': np.mean([np.sqrt(mse_u), np.sqrt(mse_v)]),
        })

    metrics_df = pd.DataFrame(rows)
    metrics_path = output_dir / 'multioutput_per_step_metrics.parquet'
    require_pyarrow()
    metrics_df.to_parquet(metrics_path, engine='pyarrow', index=False)
    print(f'Saved per-step metrics to {metrics_path}')
    return metrics_df, preds


def plot_rmse_by_lead_time(metrics_df, output_path=None, title='RMSE vs Forecast Lead Time'):
    output_path = LIGHTGBM_PLOT_DIR / 'rmse_by_lead_time.png' if output_path is None else Path(output_path)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.plot(metrics_df['lead_hours'], metrics_df['RMSE_u'], marker='s', linewidth=2, label='U-component RMSE')
    ax.plot(metrics_df['lead_hours'], metrics_df['RMSE_v'], marker='o', linewidth=2, label='V-component RMSE')
    ax.plot(metrics_df['lead_hours'], metrics_df['Avg_RMSE'], marker='x', linewidth=2.5, color='black', label='Average RMSE')
    ax.set_xlabel('Forecast Lead Time (hours)')
    ax.set_ylabel('RMSE')
    ax.set_title(title)
    ax.set_xlim(left=0, right=NUM_FORECAST_STEPS * TIMESTEP_MINUTES / 60 + 0.5)
    ax.set_ylim(bottom=0)
    ax.grid(True, linestyle=':', alpha=0.7)
    ax.legend()
    plt.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved RMSE plot to {output_path}')
    return output_path


print('Multi-output LightGBM functions ready')
# Run manually when needed:
# quick_model, quick_feature_cols, quick_target_cols = train_quick_multioutput_lightgbm(
#     train_df,
#     params=DEFAULT_QUICK_LGBM_PARAMS,
#     subsample_frac=0.1,
# )
# quick_metrics_df, quick_predictions = evaluate_multioutput_lightgbm(
#     quick_model,
#     test_df,
#     quick_feature_cols,
#     quick_target_cols,
# )
# plot_rmse_by_lead_time(quick_metrics_df)


In [ ]:
# ============================================================================
# [PART 16] Single-target Tuning and Individual Models: tune_single_model.py / train_final_individual.py
# ============================================================================

# Cell notes:
# - Defines Optuna tuning for one target and final training for individual LightGBM models.
# - Individual-model mode trains separate models for each component/lead time, e.g. `YSSY_u_forecast_t_plus_1`.
# - This mirrors `tune_single_model.py` and `train_final_individual.py`.


def tune_single_lightgbm_target(train_df, validation_df, target_col, n_trials=10, train_subsample_frac=0.1, output_dir=None):
    require_lightgbm_stack(require_optuna=True)
    output_dir = LIGHTGBM_OUTPUT_DIR / 'tuned_hyperparameters' if output_dir is None else Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    X_train, y_train_all, _, feature_cols, _ = split_lightgbm_xy(train_df)
    X_val, y_val_all, _, _, _ = split_lightgbm_xy(validation_df)
    y_train = y_train_all[target_col]
    y_val = y_val_all[target_col]

    if 0 < train_subsample_frac < 1.0:
        X_train = X_train.sample(frac=train_subsample_frac, random_state=42)
        y_train = y_train.loc[X_train.index]

    def objective(trial):
        params = {
            'objective': 'regression_l2',
            'metric': 'l2',
            'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 500),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state': 42,
            'n_jobs': -1,
            'verbose': -1,
        }
        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val[feature_cols], y_val)],
            eval_metric='l2',
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
        )
        preds = model.predict(X_val[feature_cols])
        trial.set_user_attr('best_iteration', model.best_iteration_)
        return mean_squared_error(y_val, preds)

    study = optuna.create_study(direction='minimize', study_name=f'tune_{target_col}')
    study.optimize(objective, n_trials=n_trials)

    trials_df = study.trials_dataframe().sort_values('value')
    safe_target = target_col.replace('+', 'p').replace('/', '_')
    trials_path = output_dir / f'all_trials_{safe_target}.csv'
    trials_df.to_csv(trials_path, index=False)

    best_params = study.best_params
    best_params['best_iteration_'] = study.best_trial.user_attrs.get('best_iteration')
    best_params['validation_score_mse'] = study.best_value
    best_params['_target_column_tuned'] = target_col
    best_params['_train_subsample_fraction_used'] = train_subsample_frac
    best_params['_optuna_n_trials_run'] = n_trials
    params_path = output_dir / f'best_params_{safe_target}.json'
    with open(params_path, 'w') as f:
        json.dump(best_params, f, indent=2)

    print(f'Saved tuning trials to {trials_path}')
    print(f'Saved best parameters to {params_path}')
    return best_params, study, trials_df


def train_individual_lightgbm_models(
    train_df,
    validation_df,
    components=('u', 'v'),
    steps=None,
    params=None,
    output_dir=None,
    early_stopping_rounds=50,
):
    require_lightgbm_stack()
    steps = range(1, NUM_FORECAST_STEPS + 1) if steps is None else list(steps)
    params = DEFAULT_FINAL_INDIVIDUAL_PARAMS.copy() if params is None else params.copy()
    output_dir = LIGHTGBM_MODEL_DIR / 'output_final_models' if output_dir is None else Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    X_train, y_train_all, _, feature_cols, _ = split_lightgbm_xy(train_df)
    X_val, y_val_all, _, _, _ = split_lightgbm_xy(validation_df)
    trained_models = {}

    for component in components:
        component_dir = output_dir / f'{component}_models'
        component_dir.mkdir(parents=True, exist_ok=True)
        for step in steps:
            target_col = f'{TARGET_STATION}_{component}_forecast_t_plus_{step}'
            if target_col not in y_train_all.columns:
                print(f'Skipping missing target: {target_col}')
                continue
            model = lgb.LGBMRegressor(**params)
            print(f'Training individual model: {target_col}')
            model.fit(
                X_train,
                y_train_all[target_col],
                eval_set=[(X_val[feature_cols], y_val_all[target_col])],
                eval_metric=params.get('metric', 'l2'),
                callbacks=[
                    lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False),
                    lgb.log_evaluation(period=250),
                ],
            )
            model_path = component_dir / f'{target_col}.joblib'
            save_joblib_artifact(model, model_path)
            trained_models[target_col] = model

    return trained_models, feature_cols


print('Single-target tuning and individual-model training functions ready')
# Run manually when needed:
# target_to_tune = f'{TARGET_STATION}_u_forecast_t_plus_1'
# tuned_params, tuning_study, tuning_trials_df = tune_single_lightgbm_target(
#     train_df,
#     validation_df,
#     target_to_tune,
#     n_trials=10,
#     train_subsample_frac=0.1,
# )
# individual_models, individual_feature_cols = train_individual_lightgbm_models(
#     train_df,
#     validation_df,
#     components=('u', 'v'),
#     steps=range(1, NUM_FORECAST_STEPS + 1),
# )


In [ ]:
# ============================================================================
# [PART 17] Individual Model Evaluation: evaluate_all_24hr_models.py
# ============================================================================

# Cell notes:
# - Defines evaluation helpers for a folder/dictionary of individual LightGBM models.
# - Produces per-lead-time MAE, MSE, and RMSE for U/V components and their average.
# - Mirrors the functionality of `evaluate_all_24hr_models.py`.


def load_individual_lightgbm_models(models_base_dir):
    require_lightgbm_stack()
    models_base_dir = Path(models_base_dir)
    loaded_models = {}
    feature_names = None

    for component in ['u', 'v']:
        component_dir = models_base_dir / f'{component}_models'
        if not component_dir.exists():
            print(f'Model directory not found, skipping: {component_dir}')
            continue
        for model_file in sorted(component_dir.glob('*.joblib')):
            model = joblib.load(model_file)
            loaded_models[model_file.stem] = model
            if feature_names is None:
                if hasattr(model, 'feature_name_'):
                    feature_names = list(model.feature_name_)
                elif hasattr(model, 'booster_') and hasattr(model.booster_, 'feature_name'):
                    feature_names = list(model.booster_.feature_name())

    print(f'Loaded {len(loaded_models)} individual models')
    return loaded_models, feature_names


def predict_with_individual_lightgbm_models(loaded_models, X_df, num_steps=NUM_FORECAST_STEPS):
    predictions = {}
    for step in range(1, num_steps + 1):
        for component in ['u', 'v']:
            target_col = f'{TARGET_STATION}_{component}_forecast_t_plus_{step}'
            model = loaded_models.get(target_col)
            if model is None:
                predictions[target_col] = np.full(len(X_df), np.nan)
            else:
                predictions[target_col] = model.predict(X_df)
    return pd.DataFrame(predictions)


def evaluate_individual_lightgbm_models(loaded_models, test_df, feature_cols=None, output_dir=LIGHTGBM_METRICS_DIR):
    require_lightgbm_stack()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    target_cols = get_lightgbm_target_cols(test_df)
    if feature_cols is None:
        feature_cols = get_lightgbm_feature_cols(test_df, target_cols)
    X_test = test_df[feature_cols]
    y_test = test_df[target_cols]
    predictions_df = predict_with_individual_lightgbm_models(loaded_models, X_test)
    predictions_df = predictions_df[target_cols]

    rows = []
    for step in range(1, NUM_FORECAST_STEPS + 1):
        u_col = f'{TARGET_STATION}_u_forecast_t_plus_{step}'
        v_col = f'{TARGET_STATION}_v_forecast_t_plus_{step}'
        pred_u = predictions_df[u_col]
        pred_v = predictions_df[v_col]
        has_u = not pred_u.isna().all()
        has_v = not pred_v.isna().all()
        mae_u = mean_absolute_error(y_test[u_col], pred_u) if has_u else np.nan
        mse_u = mean_squared_error(y_test[u_col], pred_u) if has_u else np.nan
        mae_v = mean_absolute_error(y_test[v_col], pred_v) if has_v else np.nan
        mse_v = mean_squared_error(y_test[v_col], pred_v) if has_v else np.nan
        rows.append({
            'lead_step': step,
            'lead_hours': step * TIMESTEP_MINUTES / 60,
            'MAE_u': mae_u,
            'MSE_u': mse_u,
            'RMSE_u': np.sqrt(mse_u) if pd.notna(mse_u) else np.nan,
            'MAE_v': mae_v,
            'MSE_v': mse_v,
            'RMSE_v': np.sqrt(mse_v) if pd.notna(mse_v) else np.nan,
            'Avg_MAE': np.nanmean([mae_u, mae_v]),
            'Avg_MSE': np.nanmean([mse_u, mse_v]),
            'Avg_RMSE': np.nanmean([np.sqrt(mse_u) if pd.notna(mse_u) else np.nan, np.sqrt(mse_v) if pd.notna(mse_v) else np.nan]),
        })

    metrics_df = pd.DataFrame(rows)
    metrics_path = output_dir / 'individual_per_step_metrics.parquet'
    require_pyarrow()
    metrics_df.to_parquet(metrics_path, engine='pyarrow', index=False)
    print(f'Saved individual model metrics to {metrics_path}')
    return metrics_df, predictions_df

print('Individual-model evaluation functions ready')


In [ ]:
# ============================================================================
# [PART 18] Forecast Sample Plots: plot_samples_24hr.py / plot_test_samples.py
# ============================================================================

# Cell notes:
# - Defines forecast-vs-actual plotting helpers from `plot_samples_24hr.py` and `plot_test_samples.py`.
# - Converts predicted U/V components to wind speed and direction for visual comparison.
# - Uses `FinalData/YSSY.txt` as the observed weather trace around each forecast anchor.


def uv_to_speed_direction(u_input, v_input):
    u_arr = np.asarray(u_input, dtype=float)
    v_arr = np.asarray(v_input, dtype=float)
    u_met = u_arr
    v_met = -v_arr
    wind_speed = np.sqrt(u_met ** 2 + v_met ** 2)
    wind_from_dir_deg = (270 - np.degrees(np.arctan2(v_met, u_met))) % 360
    wind_from_dir_deg = np.asarray(wind_from_dir_deg, dtype=float)
    wind_from_dir_deg[wind_speed < 0.1] = 0
    return wind_speed, wind_from_dir_deg


def load_yssy_observations_from_finaldata(final_data_dir=FINAL_DATA_DIR):
    yssy_file = Path(final_data_dir) / f'{TARGET_STATION}.txt'
    if not yssy_file.exists():
        raise FileNotFoundError(f'YSSY observation file not found: {yssy_file}')
    obs_df = pd.read_csv(yssy_file, parse_dates=['timestamp'], na_values=['NaN'])
    obs_df = obs_df.sort_values('timestamp').set_index('timestamp', drop=False)
    return obs_df


def predict_uv_sequence_for_row(model_or_models, row, feature_cols, individual_models=False):
    X_sample = pd.DataFrame([row[feature_cols]], columns=feature_cols)
    if individual_models:
        pred_u = np.full(NUM_FORECAST_STEPS, np.nan)
        pred_v = np.full(NUM_FORECAST_STEPS, np.nan)
        for step in range(1, NUM_FORECAST_STEPS + 1):
            u_name = f'{TARGET_STATION}_u_forecast_t_plus_{step}'
            v_name = f'{TARGET_STATION}_v_forecast_t_plus_{step}'
            if u_name in model_or_models:
                pred_u[step - 1] = model_or_models[u_name].predict(X_sample)[0]
            if v_name in model_or_models:
                pred_v[step - 1] = model_or_models[v_name].predict(X_sample)[0]
        return pred_u, pred_v

    pred_flat = model_or_models.predict(X_sample)[0]
    return pred_flat[0::2], pred_flat[1::2]


def plot_forecast_vs_actual_sample(
    timestamp_anchor,
    model_or_models,
    model_input_row,
    feature_cols,
    yssy_obs_df=None,
    individual_models=False,
    output_dir=LIGHTGBM_PLOT_DIR,
    plot_suffix='',
):
    require_lightgbm_stack()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    yssy_obs_df = load_yssy_observations_from_finaldata() if yssy_obs_df is None else yssy_obs_df
    timestamp_anchor = pd.Timestamp(timestamp_anchor)

    plot_start = timestamp_anchor - timedelta(hours=24)
    plot_end = timestamp_anchor + timedelta(hours=24)
    obs_slice = yssy_obs_df[(yssy_obs_df['timestamp'] >= plot_start) & (yssy_obs_df['timestamp'] <= plot_end)].copy()

    obs_times = obs_slice['timestamp'].values
    obs_temp = obs_slice.get('air_temp', pd.Series(np.nan, index=obs_slice.index)).values
    obs_dewp = obs_slice.get('dew_point', pd.Series(np.nan, index=obs_slice.index)).values
    obs_u = obs_slice.get('u_component', pd.Series(np.nan, index=obs_slice.index)).values
    obs_v = obs_slice.get('v_component', pd.Series(np.nan, index=obs_slice.index)).values
    obs_ws, obs_wd = uv_to_speed_direction(obs_u, obs_v)

    future_times = [timestamp_anchor + timedelta(minutes=TIMESTEP_MINUTES * i) for i in range(1, NUM_FORECAST_STEPS + 1)]
    pred_u, pred_v = predict_uv_sequence_for_row(model_or_models, model_input_row, feature_cols, individual_models=individual_models)
    pred_ws, pred_wd = uv_to_speed_direction(pred_u, pred_v)

    fig, ax1 = plt.subplots(figsize=(18, 10))
    model_label = 'Individual Models' if individual_models else 'Multi-output Model'
    fig.suptitle(f'YSSY Forecast ({model_label}) anchored at {timestamp_anchor:%Y-%m-%d %H:%M}', fontsize=16)

    ax1.plot(obs_times, obs_temp, color='red', label='Observed Temp (deg C)', zorder=3)
    ax1.plot(obs_times, obs_dewp, color='blue', label='Observed Dewpoint (deg C)', zorder=3)
    ax1.plot(obs_times, obs_ws, color='gray', linewidth=3, label='Observed Wind Speed', zorder=3)
    ax1.plot(future_times, pred_ws, color='black', linewidth=4, label='Forecast Wind Speed', zorder=4)
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Temperature / Wind Speed')
    ax1.grid(True, linestyle=':', alpha=0.7)
    ax1.set_xlim(plot_start, plot_end)
    ax1.yaxis.set_major_locator(plt.MultipleLocator(5))

    ax2 = ax1.twinx()
    ax2.scatter(obs_times, obs_wd, marker='o', color='darkgray', s=50, label='Observed Wind Direction', zorder=6)
    ax2.scatter(future_times, pred_wd, marker='X', color='black', s=70, label='Forecast Wind Direction', zorder=7)
    ax2.set_ylabel('Wind Direction (degrees from North)')
    ax2.set_ylim(0, 360)
    ax2.yaxis.set_major_locator(plt.MultipleLocator(45))

    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M\n%d-%b'))
    ax1.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax1.axvline(timestamp_anchor, color='black', linestyle=':', linewidth=1, alpha=0.7, label='Forecast Anchor')

    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    output_path = output_dir / f'forecast_{timestamp_anchor:%Y%m%d_%H%M}{plot_suffix}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved forecast plot to {output_path}')
    return output_path


def plot_random_forecast_samples(model_or_models, test_df, feature_cols, n_samples=5, individual_models=False, output_dir=LIGHTGBM_PLOT_DIR):
    if len(test_df) == 0:
        print('Test set is empty. No plots generated.')
        return []
    n_samples = min(n_samples, len(test_df))
    sample_indices = random.sample(range(len(test_df)), n_samples)
    yssy_obs_df = load_yssy_observations_from_finaldata()
    output_paths = []
    for sample_number, idx in enumerate(sample_indices, start=1):
        row = test_df.iloc[idx]
        output_paths.append(
            plot_forecast_vs_actual_sample(
                row['timestamp_t'],
                model_or_models,
                row,
                feature_cols,
                yssy_obs_df=yssy_obs_df,
                individual_models=individual_models,
                output_dir=output_dir,
                plot_suffix=f'_rand{sample_number}',
            )
        )
    return output_paths

print('Forecast plotting functions ready')


## Execute LightGBM Training

Run these cells after the Parquet datasets are available. They load the prepared datasets, train the quick multi-output LightGBM model, compare validation and test performance, and generate forecast sample plots.


In [ ]:
# ============================================================================
# [PART 19] Load Prepared Parquet Datasets for Training
# ============================================================================

# Cell notes:
# - Loads prepared training, validation, and test Parquet datasets for model training.
# - Validates that all splits use the same feature and target columns.
# - Creates `train_df`, `validation_df`, `test_df`, and LightGBM column lists for the later training cells.

prepared_datasets = load_ml_datasets_from_parquet(OUTPUT_DIR)

train_df = prepared_datasets['training']['dataframe']
validation_df = prepared_datasets['validation']['dataframe']
test_df = prepared_datasets['test']['dataframe']

X_train, y_train, ts_train, train_feature_cols, train_target_cols = split_lightgbm_xy(train_df)
X_validation, y_validation, ts_validation, validation_feature_cols, validation_target_cols = split_lightgbm_xy(validation_df)
X_test, y_test, ts_test, test_feature_cols, test_target_cols = split_lightgbm_xy(test_df)

if train_feature_cols != validation_feature_cols or train_feature_cols != test_feature_cols:
    raise ValueError('Feature columns differ between training, validation, and test datasets')
if train_target_cols != validation_target_cols or train_target_cols != test_target_cols:
    raise ValueError('Target columns differ between training, validation, and test datasets')

LIGHTGBM_FEATURE_COLS = train_feature_cols
LIGHTGBM_TARGET_COLS = train_target_cols

print('Prepared datasets loaded for LightGBM training')
print(f'  Training:   X={X_train.shape}, y={y_train.shape}')
print(f'  Validation: X={X_validation.shape}, y={y_validation.shape}')
print(f'  Test:       X={X_test.shape}, y={y_test.shape}')


In [ ]:
# ============================================================================
# [PART 20] Train Quick Multi-output LightGBM Model
# ============================================================================

# Cell notes:
# - Runs actual quick multi-output LightGBM training.
# - `TRAIN_SUBSAMPLE_FRACTION` controls how much of the training set is used; `1.0` means full training.
# - Saves the trained model as `yssy_multi_output_model_24hr_quick.joblib`.

# Training notes:
# - This cell trains a multi-output baseline: scikit-learn wraps LightGBM so each target column gets its own regressor.
# - The model predicts all 96 forecast targets: 48 lead times x 2 wind components.
# - Use this baseline first; the individual-model workflow in PART 16 is slower but gives more control per target.
# Increase subsample_frac to 1.0 for full training. A smaller value is useful for quick iteration.
TRAIN_SUBSAMPLE_FRACTION = 1.0

quick_model, quick_feature_cols, quick_target_cols = train_quick_multioutput_lightgbm(
    train_df,
    params=DEFAULT_QUICK_LGBM_PARAMS,
    subsample_frac=TRAIN_SUBSAMPLE_FRACTION,
    model_output_path=LIGHTGBM_MODEL_DIR / 'yssy_multi_output_model_24hr_quick.joblib',
)

print('Quick multi-output LightGBM training complete')


In [ ]:
# ============================================================================
# [PART 21] Validation and Test Evaluation
# ============================================================================

# Cell notes:
# - Evaluates the trained quick model on validation and test data.
# - Saves per-lead-time metrics and a validation/test comparison table.
# - Metrics include MAE, MSE, and RMSE for U, V, and average U/V performance.

validation_metrics_df, validation_predictions = evaluate_multioutput_lightgbm(
    quick_model,
    validation_df,
    quick_feature_cols,
    quick_target_cols,
    output_dir=LIGHTGBM_METRICS_DIR / 'validation',
)

test_metrics_df, test_predictions = evaluate_multioutput_lightgbm(
    quick_model,
    test_df,
    quick_feature_cols,
    quick_target_cols,
    output_dir=LIGHTGBM_METRICS_DIR / 'test',
)

metrics_comparison_df = validation_metrics_df[['lead_step', 'lead_hours', 'Avg_MAE', 'Avg_MSE', 'Avg_RMSE']].rename(
    columns={'Avg_MAE': 'Validation_Avg_MAE', 'Avg_MSE': 'Validation_Avg_MSE', 'Avg_RMSE': 'Validation_Avg_RMSE'}
).merge(
    test_metrics_df[['lead_step', 'Avg_MAE', 'Avg_MSE', 'Avg_RMSE']].rename(
        columns={'Avg_MAE': 'Test_Avg_MAE', 'Avg_MSE': 'Test_Avg_MSE', 'Avg_RMSE': 'Test_Avg_RMSE'}
    ),
    on='lead_step',
    how='inner',
)

comparison_path = LIGHTGBM_METRICS_DIR / 'validation_test_metrics_comparison.parquet'
require_pyarrow()
metrics_comparison_df.to_parquet(comparison_path, engine='pyarrow', index=False)
print(f'Saved validation/test metrics comparison to {comparison_path}')

metrics_comparison_df.head()


In [ ]:
# ============================================================================
# [PART 22] Visualise Validation vs Test Error by Forecast Lead Time
# ============================================================================

# Cell notes:
# - Visualises validation and test forecast error as lead time increases.
# - Left panel compares average RMSE; right panel compares U and V component RMSE.
# - Saves the plot as `validation_test_rmse_comparison.png`.

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
ax.plot(metrics_comparison_df['lead_hours'], metrics_comparison_df['Validation_Avg_RMSE'], marker='o', linewidth=2, label='Validation Avg RMSE')
ax.plot(metrics_comparison_df['lead_hours'], metrics_comparison_df['Test_Avg_RMSE'], marker='s', linewidth=2, label='Test Avg RMSE')
ax.set_xlabel('Forecast Lead Time (hours)')
ax.set_ylabel('Average RMSE')
ax.set_title('Average RMSE vs Forecast Lead Time')
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend()

ax = axes[1]
ax.plot(validation_metrics_df['lead_hours'], validation_metrics_df['RMSE_u'], linestyle='--', marker='o', label='Validation U RMSE')
ax.plot(validation_metrics_df['lead_hours'], validation_metrics_df['RMSE_v'], linestyle='--', marker='s', label='Validation V RMSE')
ax.plot(test_metrics_df['lead_hours'], test_metrics_df['RMSE_u'], linestyle='-', marker='o', label='Test U RMSE')
ax.plot(test_metrics_df['lead_hours'], test_metrics_df['RMSE_v'], linestyle='-', marker='s', label='Test V RMSE')
ax.set_xlabel('Forecast Lead Time (hours)')
ax.set_ylabel('RMSE')
ax.set_title('U/V RMSE vs Forecast Lead Time')
ax.grid(True, linestyle=':', alpha=0.7)
ax.legend()

plt.tight_layout()
validation_test_plot_path = LIGHTGBM_PLOT_DIR / 'validation_test_rmse_comparison.png'
plt.savefig(validation_test_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved validation/test RMSE comparison plot to {validation_test_plot_path}')


In [ ]:
# ============================================================================
# [PART 23] Forecast Sample Plots for Validation and Test Sets
# ============================================================================

# Cell notes:
# - Generates forecast-vs-actual sample plots for validation and test examples.
# - Each plot shows observed temperature/dewpoint/wind and predicted future wind speed/direction.
# - Adjust `NUM_VALIDATION_SAMPLE_PLOTS` and `NUM_TEST_SAMPLE_PLOTS` to control plot count.

# Adjust these counts if you want more or fewer plots.
NUM_VALIDATION_SAMPLE_PLOTS = 3
NUM_TEST_SAMPLE_PLOTS = 3

validation_plot_paths = plot_random_forecast_samples(
    quick_model,
    validation_df,
    quick_feature_cols,
    n_samples=NUM_VALIDATION_SAMPLE_PLOTS,
    individual_models=False,
    output_dir=LIGHTGBM_PLOT_DIR / 'validation_samples',
)

test_plot_paths = plot_random_forecast_samples(
    quick_model,
    test_df,
    quick_feature_cols,
    n_samples=NUM_TEST_SAMPLE_PLOTS,
    individual_models=False,
    output_dir=LIGHTGBM_PLOT_DIR / 'test_samples',
)

print('Validation sample plots:')
for path in validation_plot_paths:
    print(f'  {path}')

print('Test sample plots:')
for path in test_plot_paths:
    print(f'  {path}')
